# PGM end-to-end pipeline

This notebook runs **every stage** of the project in order: load raw data, preprocess, optionally explore QC plots, cluster cells, fit three Gaussian graphical model (GGM) variants, evaluate them, and record a reproducibility snapshot.

**Model sketch:** Graphical Lasso minimizes \(-\log\det\Theta + \mathrm{tr}(S\Theta) + \alpha\|\Theta\|_1\) with \(\Theta \succeq 0\). For mixture states, a weighted scatter \(S^{(k)}\) is used; the KG-biased stage blends covariance with a prior \(S' = S^{(k)} + \lambda P\) (see `src/pgm/models/`).

Artifacts land under `data/`, `reports/`, and `logs/` as configured in `configs/default.yaml`, optional presets such as `configs/quick.yaml`, and `smoke.yaml` when smoke mode is on.

## 1. Setup

The package expects to run with the **repository root** on `sys.path` and as the working directory so paths like `configs/`, `data/`, and `datasets/` resolve correctly. This cell moves to the parent of `notebooks/` and imports the pipeline entrypoints.

For **tiny** fast runs, use `SMOKE=1` or `smoke=True` (merges `configs/smoke.yaml`). For a **medium** runtime (about 1.6k cells, fewer HVGs/bootstraps/STRING genes), use `preset="quick"` or set env `PGM_PRESET=quick` (merges `configs/quick.yaml` on top of default).

In [1]:
from pathlib import Path
import json
import os
import time

os.chdir(Path("..").resolve())

FORCE_RERUN = True  # set True to ignore existing checkpoints / outputs where supported

from pgm.config.loader import load_config
from pgm.pipelines.ingest import ingest_pbmc_pipeline
from pgm.pipelines.preprocess import run_preprocess_pipeline
from pgm.pipelines.eda import run_eda
from pgm.pipelines.cluster import run_clustering_pipeline
from pgm.pipelines.global_ggm import run_global_ggm_pipeline
from pgm.pipelines.soft_ggm import run_soft_weighted_ggm_pipeline
from pgm.pipelines.kg_pipeline import run_kg_pipeline
from pgm.pipelines.evaluation import run_evaluation_pipeline
from pgm.utils.env import snapshot_run_environment


Loads `configs/default.yaml`, optionally merges `configs/{preset}.yaml` when `preset` is set (or `PGM_PRESET` is set in the environment), then merges `configs/smoke.yaml` when smoke mode is enabled. Validates into a `ProjectConfig`. All later stages read this single object so behavior is consistent.

In [2]:
# preset="quick" → mid-scale (see configs/quick.yaml). Use preset=None for full data; smoke=True for tiny runs.
cfg = load_config(smoke=False, preset="quick")
cfg


ProjectConfig(paths=PathsConfig(project_root=None), data=DataPathsConfig(tenx_mtx_dir=WindowsPath('datasets/filtered_gene_bc_matrices/hg19'), interim_dir=WindowsPath('data/interim'), processed_dir=WindowsPath('data/processed'), checkpoint_dir=WindowsPath('data/checkpoints'), external_dir=WindowsPath('data/external'), string_cache_dir=WindowsPath('data/external/string_cache')), reports=ReportsPathsConfig(figures_dir=WindowsPath('reports/figures'), eda_dir=WindowsPath('reports/eda'), results_dir=WindowsPath('reports/results')), run=RunConfig(random_seed=42), logging=LoggingConfig(level='INFO', console=True, log_dir=WindowsPath('logs'), rotating_max_bytes=10485760, rotating_backup_count=3), checkpoint=CheckpointRuntimeConfig(subdirectory='quick'), eda=EdaRuntimeConfig(n_cells_profiling_hint=1600, umap_neighbors=15, umap_min_dist=0.5), ingestion=IngestionConfig(dataset_name='pbmc3k', write_interim_filename='pbmc3k_raw.h5ad'), preprocessing=PreprocessingConfig(target_sum=10000.0, min_genes=

## 3. Data ingestion

Reads the 10x PBMC matrix from `data.tenx_mtx_dir`, performs basic validation, writes a JSON summary under `reports/results/`, and saves interim AnnData to `data/interim/` (`ingestion.write_interim_filename`). Checkpoints avoid recomputation when `FORCE_RERUN` is False.


In [3]:
t_ingest = time.perf_counter()
interim_raw_h5ad = ingest_pbmc_pipeline(cfg, force=FORCE_RERUN)
print(f"Ingestion → {interim_raw_h5ad} ({time.perf_counter() - t_ingest:.2f}s)")


Ingestion → F:\Research\PGM\data\interim\pbmc3k_raw.h5ad (0.33s)


## 4. Preprocessing

Standard single-cell workflow: QC filters (genes/cells, mitochondrial fraction), normalization to a target sum, log transform, highly variable genes, scaling, PCA. Writes `data/processed/` AnnData including `X_pca` for downstream neighbors and mixture models.


In [4]:
t_pre = time.perf_counter()
processed_h5ad = run_preprocess_pipeline(cfg, interim_raw_h5ad, force=FORCE_RERUN)
print(f"Preprocess → {processed_h5ad} ({time.perf_counter() - t_pre:.2f}s)")


2026-05-15 15:10:08 pgm.preprocessing INFO Subsampled cells: 1600 of 2700
2026-05-15 15:10:14 pgm.preprocessing INFO Running PCA n_comps=36 (nhvg=700)
2026-05-15 15:10:14 pgm.preprocessing INFO Preprocessed shape=(1598, 700); HVG count=700
2026-05-15 15:10:14 pgm.checkpoint INFO Saved AnnData checkpoint F:\Research\PGM\data\checkpoints\quick\preprocess_done.h5ad (AnnData)
2026-05-15 15:10:14 pgm.pipelines.preprocess INFO Preprocessed 6.66s → F:\Research\PGM\data\processed\pbmc3k_processed_quick.h5ad shape=(1598, 700)
Preprocess → F:\Research\PGM\data\processed\pbmc3k_processed_quick.h5ad (6.66s)


### 4b. Processed matrix sanity check

Loads the written processed `.h5ad` and prints non-finite counts, value range, and how many genes have ~zero standard deviation across cells. NaNs after scaling usually mean zero-variance genes; Graphical Lasso is sensitive to singular covariance. Preprocessing now drops those genes before PCA.

In [5]:
import scanpy as sc

from pgm.utils.adata_audit import format_matrix_summary, summarize_adata_matrix

_qc = sc.read_h5ad(processed_h5ad)
print(format_matrix_summary(summarize_adata_matrix(_qc)))
print("\nobs columns (sample):", list(_qc.obs.columns[:12]))
_qc.obs.head()

X shape: 1598 × 700
non-finite values: 0 (nan=0, +inf=0, -inf=0)
finite min/mean/max: -4.08091 / -0.0047544 / 10
genes with ~zero std (ddof=1): 0; genes with non-finite std: 0

obs columns (sample): ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes']


,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,n_genes
GAGGTACTGACACT-1,857,6.754604,1922.0,7.561642,40.426639,52.341311,64.724246,81.425598,26.0,3.295837,1.352758,857
TGGACCCTGGTACT-1,1212,7.100852,3734.0,8.225503,37.814676,51.794322,65.693626,80.931976,117.0,4.770685,3.133369,1212
CGGGACTGGAATAG-1,815,6.704414,2013.0,7.607878,42.126180,55.340288,67.411823,84.351714,68.0,4.234107,3.378043,815
GTAGCATGCACTCC-1,641,6.464588,1680.0,7.427144,46.130952,62.380952,73.750000,91.607143,41.0,3.737670,2.440476,641
TCACCTCTTCCAAG-1,1064,6.970730,2765.0,7.925158,37.287523,48.860759,61.952984,79.602170,71.0,4.276666,2.567812,1064


## 5. Exploratory data analysis (optional)

Builds QC-style figures under `reports/figures/eda/` and a short markdown summary under `reports/eda/`. This stage is **not** required for the modeling chain but helps verify preprocessing; it uses the processed AnnData path and respects smoke sampling hints from config.


In [6]:
t_eda = time.perf_counter()
eda_summary_path = run_eda(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"EDA summary → {eda_summary_path} ({time.perf_counter() - t_eda:.2f}s)")


2026-05-15 15:10:15 pgm.pipelines.eda INFO EDA: using existing X_pca from processed AnnData (skip re-normalize/log1p)


f:\Research\PGM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-15 15:10:34 pgm.pipelines.eda WARNING ydata-profiling unavailable (No module named 'ydata_profiling'); writing stub HTML
2026-05-15 15:10:34 pgm.checkpoint INFO Saved AnnData checkpoint F:\Research\PGM\data\checkpoints\quick\eda_processed_adata_preview.h5ad (AnnData)
2026-05-15 15:10:34 pgm.pipelines.eda INFO EDA complete in 20.12s
EDA summary → F:\Research\PGM\reports\eda\summary.md (20.12s)


## 6. Clustering and latent visualization

Computes nearest neighbors on PCA, Leiden clustering, UMAP embeddings, and fits a Gaussian mixture on the configured PCA representation to obtain **soft** component assignments. Saves clustered AnnData (default sibling `*_clustered.h5ad` in `data/processed/`) and Leiden/GMM diagnostic figures under `reports/figures/clustering/`.


In [7]:
t_cl = time.perf_counter()
clustered_h5ad = run_clustering_pipeline(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"Clustering → {clustered_h5ad} ({time.perf_counter() - t_cl:.2f}s)")


2026-05-15 15:10:37 pgm.clustering INFO GMM fit done; mean soft assignment entropy 0.0264
2026-05-15 15:10:39 pgm.checkpoint INFO Saved AnnData checkpoint F:\Research\PGM\data\checkpoints\quick\cluster_done.h5ad (AnnData)
2026-05-15 15:10:39 pgm.pipelines.cluster INFO Clustering done 4.21s → F:\Research\PGM\data\processed\pbmc3k_processed_quick_clustered.h5ad (n_obs=1598)
Clustering → F:\Research\PGM\data\processed\pbmc3k_processed_quick_clustered.h5ad (4.21s)


## 7. Global graphical Lasso (one graph for all cells)

Fits a **single** sparse inverse covariance (precision) matrix and adjacency from the full processed gene set (subject to model code paths). Serialized bundle under `data/checkpoints/global_ggm.joblib` for evaluation and comparison.


In [8]:
t_gg = time.perf_counter()
global_ggm_joblib = run_global_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Global GGM → {global_ggm_joblib} ({time.perf_counter() - t_gg:.2f}s)")


2026-05-15 15:10:39 pgm.pipelines.global_ggm INFO Global GGM loading AnnData F:\Research\PGM\data\processed\pbmc3k_processed_quick_clustered.h5ad
2026-05-15 15:10:39 pgm.pipelines.global_ggm INFO Global GGM AnnData read 0.052s shape=(1598, 700)
2026-05-15 15:10:39 pgm.models.glm INFO expression_dense shape=(1598, 700) storage=dense → float64 0.008s (8.5 MiB)
2026-05-15 15:10:39 pgm.models.global_ggm INFO Global GGM fitting on X shape (1598, 700) alpha=0.35
2026-05-15 15:10:39 pgm.models.glm INFO [global_ggm] empirical_covariance n=1598 p=700 0.011s
2026-05-15 15:10:39 pgm.models.glm INFO [global_ggm] covariance_target_shrink s=0.2200 tr_before=5.9858e+02 mu_tr_over_p=8.5512e-01
2026-05-15 15:10:39 pgm.models.glm INFO [global_ggm] stabilize_cov p=700 eig_min_pre_floor=1.9162e-01 ridge_cfg=2.0000e-02 floor_applied=2.0000e-02 (extra 0.0000e+00) eigvalsh=0.043s total=0.057s tr_after_floor=6.1258e+02
2026-05-15 15:10:39 pgm.models.glm INFO [global_ggm] graphical_lasso_from_covariance begin 

## 8. Soft mixture–weighted GGM (per latent state)

Uses GMM soft labels to fit **one graphical model per mixture component**, weighting cells by responsibility. Saves `data/checkpoints/soft_weighted_ggm.joblib` and component-level sparsity figures under `reports/figures/ggm/`. Compares heterogeneous structure across states.


In [11]:
print(
    "[Section 8] Starting soft mixture–weighted GGM (one Graphical Lasso per GMM state).\n"
    f"  HVGs (config): n_top_hvg={cfg.preprocessing.n_top_hvg}\n"
    f"  Mixture: gmm_n_components={cfg.clustering.gmm_n_components}\n"
    f"  Solver: gl_backend={cfg.models.gl_backend!r}  "
    f"gglasso_solver={cfg.models.gglasso_solver!r}\n"
    f"    gl_mode={cfg.models.gl_mode!r} (sklearn CD/LARS only)  "
    f"gl_alpha={cfg.models.gl_alpha}  "
    f"retry_mults={list(cfg.models.gl_alpha_retry_multipliers)}\n"
    f"  Covariance: covariance_ridge={cfg.models.covariance_ridge}  "
    f"covariance_shrinkage={cfg.models.covariance_shrinkage}\n"
    f"  Iterations: gl_max_iter={cfg.models.gl_max_iter}  gl_tol={cfg.models.gl_tol}\n"
    "  Live logs: INFO lines stream below; full detail also in logs/pgm.log.",
    flush=True,
)
t_soft = time.perf_counter()
soft_ggm_joblib = run_soft_weighted_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Soft GGM → {soft_ggm_joblib} ({time.perf_counter() - t_soft:.2f}s)")


[Section 8] Starting soft mixture–weighted GGM (one Graphical Lasso per GMM state).
  HVGs (config): n_top_hvg=700
  Mixture: gmm_n_components=5
  Solver: gl_mode='cd'  gl_alpha=0.35  retry_mults=[1.0, 2.0, 4.0, 8.0, 16.0]
  Covariance: covariance_ridge=0.02  covariance_shrinkage=0.22
  Iterations: gl_max_iter=2500  gl_tol=0.001
  Live logs: INFO lines stream below; full detail also in logs/pgm.log.
2026-05-15 15:17:20 pgm.pipelines.soft_ggm INFO Soft GGM loading AnnData F:\Research\PGM\data\processed\pbmc3k_processed_quick_clustered.h5ad
2026-05-15 15:17:20 pgm.pipelines.soft_ggm INFO Soft GGM AnnData read 0.115s shape=(1598, 700) obsm_keys=['X_gmm_proba', 'X_pca', 'X_umap']
2026-05-15 15:17:20 pgm.models.glm INFO expression_dense shape=(1598, 700) storage=dense → float64 0.021s (8.5 MiB)
[Soft GGM] fit_soft_component_graphs: K=5 matrix=1598x700 mean_assignment_entropy=0.0264 gl_mode='cd' gl_alpha=0.35 ridge=2.00e-02 shrinkage=0.220
2026-05-15 15:17:20 pgm.models.soft_ggm INFO fit_sof

FloatingPointError: Non SPD result: the system is too ill-conditioned for this solver. The system is too ill-conditioned for this solver

## 9. Knowledge-graph prior GGM (STRING)

Queries STRING (cached under `data/external/string_cache/`) for high-confidence edges among HVGs, builds a sparse prior matrix \(P\), and fits KG-biased GGMs per mixture state. Outputs `data/checkpoints/kg_soft_ggm.joblib` and related exports for evaluation against STRING-derived reference pairs.


In [ ]:
t_kg = time.perf_counter()
kg_joblib = run_kg_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"KG GGM → {kg_joblib} ({time.perf_counter() - t_kg:.2f}s)")


## 10. Evaluation

Aggregates **bootstrap stability** of the global adjacency, **STRING precision/recall** for the global graph, **degree/sparsity** summaries, and **cross-state similarity** (Jaccard / Frobenius) for soft and KG adjacencies where applicable. Writes `reports/results/metrics_summary.csv`.


In [ ]:
t_ev = time.perf_counter()
metrics_csv = run_evaluation_pipeline(
    cfg,
    clustered_h5ad,
    global_ggm_joblib,
    soft_ggm_joblib,
    kg_joblib,
    force=FORCE_RERUN,
)
print(f"Metrics → {metrics_csv} ({time.perf_counter() - t_ev:.2f}s)")


## 11. Run environment snapshot

Persists dependency versions and run metadata for reproducibility (location defined by `pgm.utils.env.snapshot_run_environment` and config paths). Use this alongside saved checkpoints for paper methods and debugging.


In [ ]:
t_env = time.perf_counter()
env_json = snapshot_run_environment(cfg, tag="full_pipeline")
print(f"Environment → {env_json} ({time.perf_counter() - t_env:.2f}s)")


## 12. Summary and exports

Collects all stage paths in one dict (same keys as `run_full_pipeline` in `src/pgm/pipelines/full_pipeline.py`), total wall time, and optionally writes a JSON summary under `reports/results/`.


In [ ]:
summary = {
    "interim_raw_h5ad": interim_raw_h5ad,
    "processed_h5ad": processed_h5ad,
    "eda_summary": eda_summary_path,
    "clustered_h5ad": clustered_h5ad,
    "global_ggm_joblib": global_ggm_joblib,
    "soft_ggm_joblib": soft_ggm_joblib,
    "kg_joblib": kg_joblib,
    "metrics_csv": metrics_csv,
    "env_json": env_json,
}
summary["runtime_s"] = time.perf_counter() - t_ingest

out_json = Path("reports/results/final_pipeline_summary.json")
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(
    json.dumps({k: str(v) for k, v in summary.items()}, indent=2),
    encoding="utf-8",
)
print(f"Wrote {out_json}")
summary
